In [1]:
from joblib import Parallel, delayed
from tqdm.notebook import tqdm

import sys
import pickle
import numpy as np
import pandas as pd
import networkx as nx

In [2]:
sys.path.insert(0, "src")

from grn import grn

with open("../../grns/graph.647.gpickle", "rb") as f:
    G = pickle.load(f)

# Lets work with module 8

In [3]:
module_id = 8

genes_in_module = [
    gene
    for gene, group in G.groups.items()
    if group == module_id
]

genes_in_module = [int(g) for g in genes_in_module]

print(len(genes_in_module))
print(genes_in_module[:20])

195
[6, 11, 15, 17, 20, 28, 33, 46, 68, 69, 85, 91, 106, 111, 117, 121, 128, 130, 131, 137]


In [40]:
from itertools import combinations

all_pairs_module = list(combinations(genes_in_module, 2))

In [41]:
n = 1000
rng = np.random.default_rng(42)

idx = rng.choice(len(all_pairs_module), size=n, replace=False)
random_pairs = [all_pairs_module[i] for i in idx]

In [4]:
n_jobs = 4

def run_single_ko(i):
    return G.ko_nodes(
        [i],
        stats=('new_rna',)
    )['new_rna'].flatten()


single_ko = np.array(
    Parallel(
        n_jobs=n_jobs,
        pre_dispatch='n_jobs',
        prefer='processes'
    )(
        delayed(run_single_ko)(i)
        for i in tqdm(range(G.n))
    )
)

  0%|          | 0/2000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [9]:
import torch
import torch.nn as nn

class PerturbationMLP(nn.Module):
    def __init__(self, n_genes=256):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(n_genes, 512),
            nn.GELU(),

            nn.Linear(512, 512),
            nn.GELU(),

            nn.Linear(512, n_genes)
        )

    def forward(self, x):
        return self.net(x)